In [21]:
import pandas as pd

df = pd.read_csv("data/heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [22]:
X=df.drop("target", axis=1)
y=df["target"]

In [23]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2)


In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [25]:
import torch
import torch.nn as nn

class HeartDiseaseModel(nn.Module):
    def __init__(self,input_dim):
        super(HeartDiseaseModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 32)
        self.layer2 = nn.Linear(32, 16)
        self.output_layer = nn.Linear(16, 1)
    
    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = torch.relu(self.layer2(x))
        x = torch.sigmoid(self.output_layer(x))
        return x


In [26]:
input_dim = X_train_scaled.shape[1]
model = HeartDiseaseModel(input_dim)
print(model)

HeartDiseaseModel(
  (layer1): Linear(in_features=13, out_features=32, bias=True)
  (layer2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=1, bias=True)
)


In [27]:
epochs = 100
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

In [28]:
x_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

from tqdm import tqdm

for epoch in tqdm(range(epochs)):
    model.train()
    optimizer.zero_grad()
    
    outputs = model(x_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

 39%|███▉      | 39/100 [00:00<00:00, 317.09it/s]

Epoch [10/100], Loss: 0.6235
Epoch [20/100], Loss: 0.5055
Epoch [30/100], Loss: 0.3960
Epoch [40/100], Loss: 0.3238
Epoch [50/100], Loss: 0.2829
Epoch [60/100], Loss: 0.2558
Epoch [70/100], Loss: 0.2321
Epoch [80/100], Loss: 0.2100


100%|██████████| 100/100 [00:00<00:00, 398.84it/s]

Epoch [90/100], Loss: 0.1900
Epoch [100/100], Loss: 0.1707


In [29]:
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        x_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
        
        outputs = model(x_test_tensor)
        predicted = (outputs > 0.5).float()
        
        accuracy = (predicted == y_test_tensor).float().mean()
        print(f'Accuracy: {accuracy.item():.4f}')


In [30]:
evaluate_model(model, X_test_scaled, y_test)

Accuracy: 0.8033
